# Fish + InceptionTimeClassifier: untuned vs tuned comparison

## InceptionTime-specific note

`InceptionTimeClassifier` is a deep-learning classifier. It can be much heavier than MiniRocket, WEASEL, or KNN. For a first check, use a small smoke test, for example `N_TRIALS = "2"` and a narrow window range. After verifying that the pipeline works, increase the number of trials.


In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import sys
import subprocess

## 1. General configuration


In [2]:
def find_repo_root(start_path: Path = Path.cwd()) -> Path:
    current = start_path.resolve()

    while current != current.parent:
        if (current / "scripts" / "run_experiment.py").exists():
            return current
        current = current.parent

    raise FileNotFoundError(
        "Could not find repository root containing scripts/run_experiment.py"
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

print("Working directory:", Path.cwd())
assert Path("scripts/run_experiment.py").exists()

Working directory: /Users/gabbo/Documents/GitHub/Thesis_SlidingWindow


In [3]:
assert Path("scripts/run_experiment.py").exists(), "Run the notebook from the Thesis_SlidingWindow repository root"

DATASET_FOLDER = "data/raw/Fish"
CLASSIFIER = "InceptionTimeClassifier"

WINDOW_START = "0.10"
WINDOW_END = "0.13"
WINDOW_STEP = "0.02"

STRIDE_RATIOS = ["0.1", "0.2", "0.5"]

METRIC = "series_macro_f1"
N_TRIALS = "1"
RANDOM_STATE = "42"

DATASET_NAME = Path(DATASET_FOLDER).name
WINDOW_TAG = f"{WINDOW_START}_{WINDOW_END}_step_{WINDOW_STEP}"
STRIDE_TAG = "_".join(STRIDE_RATIOS)
CONFIG_TAG = f"{WINDOW_TAG}_strides_{STRIDE_TAG}"

RESULTS_ROOT = Path("results") / DATASET_NAME / CLASSIFIER
PLOTS_ROOT = Path("plots") / DATASET_NAME / CLASSIFIER

NOT_TUNED_DIR = RESULTS_ROOT / "noTuning" / f"notTuned_{CONFIG_TAG}"
TUNING_DIR = RESULTS_ROOT / "tuning" / f"per_config_{CONFIG_TAG}"
TUNED_DIR = RESULTS_ROOT / "tuned" / f"tunedBest_{CONFIG_TAG}"

COMPARISON_DIR = RESULTS_ROOT / f"comparison_{CONFIG_TAG}"
PLOTS_DIR = PLOTS_ROOT / f"macro_f1_{CONFIG_TAG}"


for directory in [NOT_TUNED_DIR, TUNING_DIR, TUNED_DIR, COMPARISON_DIR, PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_FOLDER)
print("Classifier:", CLASSIFIER)
print("Window range:", WINDOW_START, WINDOW_END, WINDOW_STEP)
print("Stride ratios:", STRIDE_RATIOS)
print("Metric:", METRIC)
print("N trials:", N_TRIALS)


Dataset: data/raw/Fish
Classifier: InceptionTimeClassifier
Window range: 0.10 0.13 0.02
Stride ratios: ['0.1', '0.2', '0.5']
Metric: series_macro_f1
N trials: 1


In [4]:
PYTHON_EXECUTABLE = sys.executable

def run_command(command: list[str]) -> None:
    command = command.copy()

    if command[0] == "python":
        command[0] = PYTHON_EXECUTABLE
        command.insert(1, "-u")  # unbuffered output

    print("\nRunning command:")
    print(" ".join(command))
    print()

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    subprocess.run(
        command,
        check=True,
        env=env,
    )


## 3. Untuned run

Output:

```text
results/Fish/InceptionTimeClassifier/notTuned_10_50/
```


In [5]:
run_command([
    "python", "scripts/run_experiment.py",
    "--dataset-folder", DATASET_FOLDER,
    "--classifier", CLASSIFIER,
    "--window-sizes", WINDOW_START, WINDOW_END, WINDOW_STEP,
    "--percentages",
    "--stride-ratios", *STRIDE_RATIOS,
    "--output-dir", str(NOT_TUNED_DIR),
    "--random-state", RANDOM_STATE,
])



Running command:
/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/bin/python -u scripts/run_experiment.py --dataset-folder data/raw/Fish --classifier InceptionTimeClassifier --window-sizes 0.10 0.13 0.02 --percentages --stride-ratios 0.1 0.2 0.5 --output-dir results/Fish/InceptionTimeClassifier/noTuning/notTuned_0.10_0.13_step_0.02_strides_0.1_0.2_0.5 --random-state 42



Fish / InceptionTimeClassifier:   0%|          | 0/6 [12:19<?, ?config/s, stride=4, stride_ratio=0.1, window_size=46]
Traceback (most recent call last):
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/scripts/run_experiment.py", line 121, in <module>
    main()
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/scripts/run_experiment.py", line 99, in main
    results_df = run_experiment(
                 ^^^^^^^^^^^^^^^
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/src/sliding_window_tsc/experiment.py", line 150, in run_experiment
    clf.fit(X_train_w, y_train_w)
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/lib/python3.12/site-packages/aeon/utils/decorators/method_timer.py", line 42, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/lib/python3.12/site-packages/aeon/classification/base.py", line 112, in fit
    self._fit(X, y)
  File "/Users/g

KeyboardInterrupt: 

## 4. Check untuned results


In [ ]:
not_tuned_files = sorted(NOT_TUNED_DIR.glob("*.csv"))
assert not_tuned_files, f"No CSV files found in {NOT_TUNED_DIR}"

not_tuned_file = not_tuned_files[-1]
not_tuned = pd.read_csv(not_tuned_file)

print("Untuned file:", not_tuned_file)
print()
print("Status counts:")
print(not_tuned["status"].value_counts())
print()

display(not_tuned[[
    "dataset",
    "classifier",
    "window_size",
    "window_percentage",
    "stride_ratio",
    "series_macro_f1",
    "status",
    "error",
]].head(20))


Untuned file: results/Fish/InceptionTimeClassifier/noTuning/notTuned_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/Fish_InceptionTimeClassifier_20260707_084931.csv

Status counts:
status
failed    6
Name: count, dtype: int64



,dataset,classifier,window_size,window_percentage,stride_ratio,series_macro_f1,status,error
0,Fish,InceptionTimeClassifier,46,0.099352,0.1,NaN,failed,InceptionTimeClassifier requires package 'tens...
1,Fish,InceptionTimeClassifier,46,0.099352,0.2,NaN,failed,InceptionTimeClassifier requires package 'tens...
2,Fish,InceptionTimeClassifier,46,0.099352,0.5,NaN,failed,InceptionTimeClassifier requires package 'tens...
3,Fish,InceptionTimeClassifier,55,0.118790,0.1,NaN,failed,InceptionTimeClassifier requires package 'tens...
4,Fish,InceptionTimeClassifier,55,0.118790,0.2,NaN,failed,InceptionTimeClassifier requires package 'tens...
5,Fish,InceptionTimeClassifier,55,0.118790,0.5,NaN,failed,InceptionTimeClassifier requires package 'tens...


## 5. Tuning for each window/stride combination

This command searches for the best hyperparameters separately for each combination:

```text
window_size × stride_ratio
```

Main outputs:

```text
results/tuning/Fish/InceptionTimeClassifier/per_config_10_50/best_hyperparameters/
results/tuning/Fish/InceptionTimeClassifier/per_config_10_50/best_hyperparameters_index.csv
```

In [ ]:
run_command([
    "python", "scripts/tune_per_configuration.py",
    "--dataset-folder", DATASET_FOLDER,
    "--classifier", CLASSIFIER,
    "--window-sizes", WINDOW_START, WINDOW_END, WINDOW_STEP,
    "--percentages",
    "--stride-ratios", *STRIDE_RATIOS,
    "--metric", METRIC,
    "--n-trials", N_TRIALS,
    "--output-dir", str(TUNING_DIR),
    "--random-state", RANDOM_STATE,
])



Running command:
/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/bin/python -u scripts/tune_per_configuration.py --dataset-folder data/raw/Fish --classifier InceptionTimeClassifier --window-sizes 0.10 0.13 0.02 --percentages --stride-ratios 0.1 0.2 0.5 --metric series_macro_f1 --n-trials 1 --output-dir results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5 --random-state 42


[1/6] Tuning Fish / InceptionTimeClassifier / window_size=46 / stride_ratio=0.1


[I 2026-07-07 08:49:32,600] A new study created in memory with name: no-name-c3099211-da6a-4696-b82e-18d38565070c
Fish / InceptionTimeClassifier: 100%|██████████| 1/1 [00:00<00:00,  3.67config/s, status=failed, stride=4, stride_ratio=0.1, window_size=46]
[I 2026-07-07 08:49:32,894] Trial 0 finished with value: 0.0 and parameters: {'n_classifiers': 5, 'n_filters': 32, 'kernel_size': 60, 'depth': 4, 'n_conv_per_layer': 2, 'bottleneck_size': 64, 'use_residual': False, 'use_bottleneck': False, 'use_max_pooling': False, 'batch_size': 64, 'n_epochs': 100}. Best is trial 0 with value: 0.0.
[I 2026-07-07 08:49:32,898] A new study created in memory with name: no-name-f04c9d8b-450e-429b-9cf7-02df655e2829
Fish / InceptionTimeClassifier: 100%|██████████| 1/1 [00:00<00:00, 48.16config/s, status=failed, stride=9, stride_ratio=0.2, window_size=46]
[I 2026-07-07 08:49:32,934] Trial 0 finished with value: 0.0 and parameters: {'n_classifiers': 5, 'n_filters': 16, 'kernel_size': 60, 'depth': 4, 'n_conv_p


[2/6] Tuning Fish / InceptionTimeClassifier / window_size=46 / stride_ratio=0.2

[3/6] Tuning Fish / InceptionTimeClassifier / window_size=46 / stride_ratio=0.5

[4/6] Tuning Fish / InceptionTimeClassifier / window_size=55 / stride_ratio=0.1

[5/6] Tuning Fish / InceptionTimeClassifier / window_size=55 / stride_ratio=0.2

[6/6] Tuning Fish / InceptionTimeClassifier / window_size=55 / stride_ratio=0.5

Tuning completed.
Saved 6 best hyperparameter files.
Index file: results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters_index.csv


## 6. Check tuning


In [ ]:
index_file = TUNING_DIR / "best_hyperparameters_index.csv"
assert index_file.exists(), f"Index file not found: {index_file}"

best_index = pd.read_csv(index_file)

print("Best hyperparameters index:", index_file)
print()
print("Best status counts:")
print(best_index["best_status"].value_counts(dropna=False))
print()

display(best_index[[
    "window_size",
    "window_percentage",
    "stride_ratio",
    "best_value",
    "best_params",
    "best_status",
    "best_error",
    "best_file",
]].head(20))


Best hyperparameters index: results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters_index.csv

Best status counts:
best_status
failed    6
Name: count, dtype: int64



,window_size,window_percentage,stride_ratio,best_value,best_params,best_status,best_error,best_file
0,46,0.099352,0.1,0.0,"{""batch_size"": 64, ""bottleneck_size"": 64, ""dep...",failed,InceptionTimeClassifier requires package 'tens...,results/Fish/InceptionTimeClassifier/tuning/pe...
1,46,0.099352,0.2,0.0,"{""batch_size"": 64, ""bottleneck_size"": 32, ""dep...",failed,InceptionTimeClassifier requires package 'tens...,results/Fish/InceptionTimeClassifier/tuning/pe...
2,46,0.099352,0.5,0.0,"{""batch_size"": 32, ""bottleneck_size"": 16, ""dep...",failed,InceptionTimeClassifier requires package 'tens...,results/Fish/InceptionTimeClassifier/tuning/pe...
3,55,0.118790,0.1,0.0,"{""batch_size"": 16, ""bottleneck_size"": 64, ""dep...",failed,InceptionTimeClassifier requires package 'tens...,results/Fish/InceptionTimeClassifier/tuning/pe...
4,55,0.118790,0.2,0.0,"{""batch_size"": 16, ""bottleneck_size"": 64, ""dep...",failed,InceptionTimeClassifier requires package 'tens...,results/Fish/InceptionTimeClassifier/tuning/pe...
5,55,0.118790,0.5,0.0,"{""batch_size"": 32, ""bottleneck_size"": 16, ""dep...",failed,InceptionTimeClassifier requires package 'tens...,results/Fish/InceptionTimeClassifier/tuning/pe...


## 7. Regenerate results using the best hyperparameters

This step loads all JSON files saved in:

```text
best_hyperparameters/
```

and reruns the final experiments using the best hyperparameters for each configuration.


In [ ]:
best_dir = TUNING_DIR / "best_hyperparameters"
assert best_dir.exists(), f"Best hyperparameter directory not found: {best_dir}"

run_command([
    "python", "scripts/run_best_hyperparameters.py",
    "--best-dir", str(best_dir),
    "--data-root", "data/raw",
    "--output-dir", str(TUNED_DIR),
    "--random-state", RANDOM_STATE,
    "--skip-failed-best",
])



Running command:
/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/bin/python -u scripts/run_best_hyperparameters.py --best-dir results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters --data-root data/raw --output-dir results/Fish/InceptionTimeClassifier/tuned/tunedBest_0.10_0.13_step_0.02_strides_0.1_0.2_0.5 --random-state 42 --skip-failed-best

Skipping failed best configuration: results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters/Fish_InceptionTimeClassifier_window_46_stride_0p1.json
Skipping failed best configuration: results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters/Fish_InceptionTimeClassifier_window_46_stride_0p2.json
Skipping failed best configuration: results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters/Fish_InceptionTim

Traceback (most recent call last):
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/scripts/run_best_hyperparameters.py", line 121, in <module>
    main()
  File "/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/scripts/run_best_hyperparameters.py", line 107, in main
    raise ValueError("No experiments were executed.")
ValueError: No experiments were executed.


CalledProcessError: Command '['/Users/gabbo/Documents/GitHub/Thesis_SlidingWindow/.venv/bin/python', '-u', 'scripts/run_best_hyperparameters.py', '--best-dir', 'results/Fish/InceptionTimeClassifier/tuning/per_config_0.10_0.13_step_0.02_strides_0.1_0.2_0.5/best_hyperparameters', '--data-root', 'data/raw', '--output-dir', 'results/Fish/InceptionTimeClassifier/tuned/tunedBest_0.10_0.13_step_0.02_strides_0.1_0.2_0.5', '--random-state', '42', '--skip-failed-best']' returned non-zero exit status 1.

## 8. Check tuned results


In [ ]:
tuned_files = sorted(TUNED_DIR.glob("*.csv"))
assert tuned_files, f"No CSV files found in {TUNED_DIR}"

tuned_file = tuned_files[-1]
tuned = pd.read_csv(tuned_file)

print("Tuned file:", tuned_file)
print()
print("Status counts:")
print(tuned["status"].value_counts())
print()

display(tuned[[
    "dataset",
    "classifier",
    "window_size",
    "window_percentage",
    "stride_ratio",
    "series_macro_f1",
    "status",
    "error",
    "classifier_hyperparameters",
]].head(20))


## 9. Create the comparison CSV

The comparison is performed using the following keys:

```text
dataset
classifier
window_size
stride_ratio
```

The main metric is:

```text
series_macro_f1
```


In [ ]:
not_tuned_ok = not_tuned[not_tuned["status"] == "ok"].copy()
tuned_ok = tuned[tuned["status"] == "ok"].copy()

not_tuned_small = not_tuned_ok[[
    "dataset",
    "classifier",
    "window_size",
    "window_percentage",
    "stride_ratio",
    "series_macro_f1",
    "fit_time_sec",
    "predict_time_sec",
    "total_time_sec",
]].rename(columns={
    "series_macro_f1": "not_tuned_series_macro_f1",
    "fit_time_sec": "not_tuned_fit_time_sec",
    "predict_time_sec": "not_tuned_predict_time_sec",
    "total_time_sec": "not_tuned_total_time_sec",
})

tuned_small = tuned_ok[[
    "dataset",
    "classifier",
    "window_size",
    "window_percentage",
    "stride_ratio",
    "series_macro_f1",
    "fit_time_sec",
    "predict_time_sec",
    "total_time_sec",
    "classifier_hyperparameters",
]].rename(columns={
    "series_macro_f1": "tuned_series_macro_f1",
    "fit_time_sec": "tuned_fit_time_sec",
    "predict_time_sec": "tuned_predict_time_sec",
    "total_time_sec": "tuned_total_time_sec",
})

comparison = not_tuned_small.merge(
    tuned_small,
    on=["dataset", "classifier", "window_size", "window_percentage", "stride_ratio"],
    how="inner",
)

comparison["delta_series_macro_f1"] = (
    comparison["tuned_series_macro_f1"]
    - comparison["not_tuned_series_macro_f1"]
)

comparison = comparison.sort_values(["stride_ratio", "window_size"])

comparison_file = COMPARISON_DIR / "fish_inceptiontime_not_tuned_vs_tuned_macro_f1.csv"
comparison.to_csv(comparison_file, index=False)

print("Saved comparison to:", comparison_file)
print()
display(comparison.head(20))


## 10. Create a long-format CSV for the tuned vs untuned plot



In [ ]:
plot_not_tuned = not_tuned_ok.copy()
plot_not_tuned["run_type"] = "Untuned"

plot_tuned = tuned_ok.copy()
plot_tuned["run_type"] = "Tuned"

plot_df = pd.concat([plot_not_tuned, plot_tuned], ignore_index=True)
plot_df = plot_df.sort_values(["stride_ratio", "window_size", "run_type"])

plot_input_file = COMPARISON_DIR / "fish_inceptiontime_plot_input_macro_f1.csv"
plot_df.to_csv(plot_input_file, index=False)

print("Saved plot input to:", plot_input_file)
display(plot_df[[
    "run_type",
    "window_size",
    "window_percentage",
    "stride_ratio",
    "series_macro_f1",
    "status",
]].head(20))


## 11. Plot: Macro F1 untuned vs tuned


In [ ]:
for stride_ratio, group in plot_df.groupby("stride_ratio"):
    fig, ax = plt.subplots(figsize=(12, 6))

    for run_type, run_group in group.groupby("run_type"):
        run_group = run_group.sort_values("window_percentage")
        ax.plot(
            run_group["window_percentage"] * 100,
            run_group["series_macro_f1"],
            marker="o",
            label=run_type,
        )

    ax.set_title(f"Fish - InceptionTimeClassifier - Macro F1 | stride_ratio={stride_ratio}")
    ax.set_xlabel("Window size (%)")
    ax.set_ylabel("Series macro F1")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Run type")

    fig.tight_layout()

    output_file = PLOTS_DIR / f"macro_f1_not_tuned_vs_tuned_stride_{str(stride_ratio).replace('.', 'p')}.png"
    fig.savefig(output_file, dpi=300, bbox_inches="tight")

    print("Saved:", output_file)
    plt.show()


In [ ]:
# ---------------------------------------------------------------------
# Final cell: total execution time and per-configuration training times
# ---------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

TIME_PLOTS_DIR = PLOTS_DIR / "execution_time"
TIME_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Reload data if needed
# ---------------------------------------------------------------------

if "not_tuned" not in globals():
    not_tuned_file = sorted(NOT_TUNED_DIR.glob("*.csv"))[-1]
    not_tuned = pd.read_csv(not_tuned_file)

if "tuned" not in globals():
    tuned_file = sorted(TUNED_DIR.glob("*.csv"))[-1]
    tuned = pd.read_csv(tuned_file)

not_tuned_ok = not_tuned[not_tuned["status"] == "ok"].copy()
tuned_ok = tuned[tuned["status"] == "ok"].copy()

# ---------------------------------------------------------------------
# 1. Total time of each phase
# ---------------------------------------------------------------------
# Untuned phase:
#   one full run with default hyperparameters.
#
# Tuning phase:
#   sum of all Optuna trial durations saved in the trials CSV files.
#
# Tuned retraining phase:
#   final rerun using the best hyperparameters for each window/stride config.
# ---------------------------------------------------------------------

untuned_total_time_sec = not_tuned_ok["total_time_sec"].sum()
tuned_retraining_total_time_sec = tuned_ok["total_time_sec"].sum()

# Read Optuna trial durations
trials_dir = TUNING_DIR / "trials"
trial_files = sorted(trials_dir.glob("*_trials.csv"))

if len(trial_files) == 0:
    raise FileNotFoundError(f"No Optuna trial CSV files found in: {trials_dir}")

tuning_total_time_sec = 0.0
trial_rows = []

for trial_file in trial_files:
    trials_df = pd.read_csv(trial_file)

    if "duration" not in trials_df.columns:
        raise ValueError(
            f"Column 'duration' not found in {trial_file}. "
            "Cannot compute total tuning time."
        )

    durations_sec = pd.to_timedelta(trials_df["duration"]).dt.total_seconds()
    file_total_sec = durations_sec.sum()

    tuning_total_time_sec += file_total_sec

    trial_rows.append({
        "trial_file": str(trial_file),
        "n_trials": len(trials_df),
        "total_duration_sec": file_total_sec,
    })

trial_time_summary = pd.DataFrame(trial_rows)

phase_time_summary = pd.DataFrame([
    {
        "phase": "Untuned experiment",
        "total_time_sec": untuned_total_time_sec,
        "total_time_min": untuned_total_time_sec / 60,
        "total_time_hour": untuned_total_time_sec / 3600,
    },
    {
        "phase": "Hyperparameter tuning",
        "total_time_sec": tuning_total_time_sec,
        "total_time_min": tuning_total_time_sec / 60,
        "total_time_hour": tuning_total_time_sec / 3600,
    },
    {
        "phase": "Tuned retraining",
        "total_time_sec": tuned_retraining_total_time_sec,
        "total_time_min": tuned_retraining_total_time_sec / 60,
        "total_time_hour": tuned_retraining_total_time_sec / 3600,
    },
])

display(phase_time_summary)
display(trial_time_summary.head())

phase_time_file = COMPARISON_DIR / "fish_inceptiontime_phase_total_times.csv"
trial_time_file = COMPARISON_DIR / "fish_inceptiontime_tuning_trial_total_times.csv"

phase_time_summary.to_csv(phase_time_file, index=False)
trial_time_summary.to_csv(trial_time_file, index=False)

print("Saved phase time summary to:", phase_time_file)
print("Saved tuning trial time summary to:", trial_time_file)

# Plot total phase times
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    phase_time_summary["phase"],
    phase_time_summary["total_time_min"],
)

ax.set_title("Fish - InceptionTimeClassifier - Total execution time by phase")
ax.set_xlabel("Phase")
ax.set_ylabel("Total time (minutes)")
ax.grid(axis="y", alpha=0.3)

for index, row in phase_time_summary.iterrows():
    ax.text(
        index,
        row["total_time_min"],
        f'{row["total_time_min"]:.2f} min',
        ha="center",
        va="bottom",
    )

fig.tight_layout()

output_file = TIME_PLOTS_DIR / "total_execution_time_by_phase.png"
fig.savefig(output_file, dpi=300, bbox_inches="tight")

print("Saved:", output_file)
plt.show()

# ---------------------------------------------------------------------
# 2. Per-configuration training time: untuned
# ---------------------------------------------------------------------

untuned_plot = not_tuned_ok.sort_values(["stride_ratio", "window_percentage"]).copy()

fig, ax = plt.subplots(figsize=(12, 6))

for stride_ratio, group in untuned_plot.groupby("stride_ratio"):
    group = group.sort_values("window_percentage")

    ax.plot(
        group["window_percentage"] * 100,
        group["fit_time_sec"],
        marker="o",
        label=f"stride_ratio={stride_ratio}",
    )

ax.set_title("Fish - InceptionTimeClassifier - Untuned training time by configuration")
ax.set_xlabel("Window size (%)")
ax.set_ylabel("Fit time (seconds)")
ax.grid(True, alpha=0.3)
ax.legend(title="Stride ratio")

fig.tight_layout()

output_file = TIME_PLOTS_DIR / "untuned_fit_time_by_configuration.png"
fig.savefig(output_file, dpi=300, bbox_inches="tight")

print("Saved:", output_file)
plt.show()

# ---------------------------------------------------------------------
# 3. Per-configuration training time: tuned retraining
# ---------------------------------------------------------------------

tuned_plot = tuned_ok.sort_values(["stride_ratio", "window_percentage"]).copy()

fig, ax = plt.subplots(figsize=(12, 6))

for stride_ratio, group in tuned_plot.groupby("stride_ratio"):
    group = group.sort_values("window_percentage")

    ax.plot(
        group["window_percentage"] * 100,
        group["fit_time_sec"],
        marker="o",
        label=f"stride_ratio={stride_ratio}",
    )

ax.set_title("Fish - InceptionTimeClassifier - Tuned retraining time by configuration")
ax.set_xlabel("Window size (%)")
ax.set_ylabel("Fit time (seconds)")
ax.grid(True, alpha=0.3)
ax.legend(title="Stride ratio")

fig.tight_layout()

output_file = TIME_PLOTS_DIR / "tuned_fit_time_by_configuration.png"
fig.savefig(output_file, dpi=300, bbox_inches="tight")

print("Saved:", output_file)
plt.show()